# Social Network Analysis of a Facebook ego-network

Structural analysis of the Stanford SNAP Facebook combined ego-network (4,039 users, 88,234
friendships): basic graph statistics, edge betweenness to find the bridges that hold the
network together, and a comparison of Louvain and Label Propagation community detection.

The edge list is included in `data/` (see `data/README.md`).

### Imports and global settings

In [ ]:
# Standard scientific stack
import os
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Network analysis
import networkx as nx

# Louvain community detection (python-louvain package, imported as 'community')
import community as community_louvain

# Built-in NetworkX label propagation
from networkx.algorithms.community import label_propagation_communities

# For comparing two partitions of the same node set
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

In [ ]:
# Path to the dataset and folder for saving figures
DATA_PATH = "../data/facebook_combined.txt"

# Silence non-essential warnings to keep the output readable
warnings.filterwarnings("ignore")

# Fixed seed so the random parts (sampling, layout, label propagation) are reproducible across runs
SEED = 0
random.seed(SEED)
np.random.seed(SEED)

# Cleaner default plot style
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150})

# Create a folder where all the figures I generate will be saved
os.makedirs('figures', exist_ok=True)

## Task 1: Build the network and report basic characteristics

The dataset is just an edge list: each line gives two user IDs that are connected by a Facebook friendship. Friendships are undirected (if A is a
friend of B then B is a friend of A) and unweighted, so a plain undirected graph is the right object.

In [ ]:
def load_graph(path):
    """Read the SNAP edge list into an undirected NetworkX graph."""
    # Quick sanity check before trying to read the file
    if not os.path.exists(path):
        raise FileNotFoundError(f'Could not find {path} — put it next to this notebook.')

    # nodetype=int keeps the original numeric IDs as ints (not strings)
    G = nx.read_edgelist(path, nodetype=int)
    print(f'Loaded: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
    return G


G = load_graph(DATA_PATH)

In [ ]:
def basic_stats(G):
    """Return the basic structural metrics asked for in Task 1, plus a couple
    of useful extras (clustering coefficient and connectedness)."""
    n = G.number_of_nodes()
    m = G.number_of_edges()

    # Average degree = 2|E| / |V| because each edge contributes to two nodes
    avg_deg = 2 * m / n

    # Density = actual edges / possible edges in an undirected simple graph
    density = nx.density(G)

    return pd.Series({
        'Nodes |V|': n,
        'Edges |E|': m,
        'Density': round(density, 6),
        'Average degree': round(avg_deg, 2),
        'Connected': nx.is_connected(G),
        'Average clustering coefficient': round(nx.average_clustering(G), 3),
    }, name='value')


basic_stats(G)

The graph is a single connected component, so no nodes are isolated. The density is around 1% and this  is what we expect for a real friendship network.

### Degree distribution

Before drawing the whole network, it helps to look at the degree distribution (how many friends each user has). Real social networks tend to be heavy-tailed, with a few hubs that have far more connections than the typical user. The log-log plot is the standard way to see that shape.

In [ ]:
def plot_degree_distribution(G, save_as=None):
    """Side-by-side linear histogram and log-log scatter of node degrees."""
    # Pull the degree of every node into a numpy array
    degrees = np.array([d for _, d in G.degree()])
    print(f'Degree — min: {degrees.min()}, median: {int(np.median(degrees))}, max: {degrees.max()}')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    # Left panel: linear histogram (shows the bulk of the distribution)
    axes[0].hist(degrees, bins=60, color='#3b6ea5', edgecolor='white')
    axes[0].set_xlabel('Degree')
    axes[0].set_ylabel('Number of nodes')
    axes[0].set_title('Linear scale')

    # Right panel: log-log scatter (shows the tail)
    counts = Counter(degrees.tolist())
    x_vals, y_vals = zip(*sorted(counts.items()))
    axes[1].scatter(x_vals, y_vals, s=12, color='#3b6ea5', alpha=0.8)
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Degree (log)')
    axes[1].set_ylabel('Frequency (log)')
    axes[1].set_title('Log–log scale')

    plt.suptitle('Degree distribution of the Facebook ego-network')
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as)
    plt.show()
    return degrees


degrees = plot_degree_distribution(G, save_as='figures/fig1_degree.png')

The log-log plot has a heavy tail: most users have only a handful of friends (median around 25), but a small number of hubs reach into the hundreds.

### Network visualisation

For drawing the network I use a Fruchterman–Reingold (spring) layout. Node size and colour are scaled by degree so the hub structure is easy to spot.

In [ ]:
def visualise_network(G, seed=SEED, save_as=None):
    """
    Draw the whole graph using a spring layout. 
    Returns the layout dict so it can be reused for later plots (community colouring, bridges).
    """
    # k controls how strongly nodes repel each other. 0.15 gives readable blobs without too much overlap on this graph
    pos = nx.spring_layout(G, seed=seed, k=0.15, iterations=20)

    # Size the markers by degree. hubs become visually prominent
    deg = dict(G.degree())
    node_size = [max(2, 0.05 * deg[n]) for n in G.nodes()]
    node_color = [deg[n] for n in G.nodes()]

    plt.figure(figsize=(10, 9))
    # Edges drawn first (very faint) so they don't dominate
    nx.draw_networkx_edges(G, pos, alpha=0.05, width=0.4)
    nx.draw_networkx_nodes(
        G, pos,
        node_size=node_size,
        node_color=node_color,
        cmap='viridis',
        linewidths=0,
    )
    plt.axis('off')
    plt.title(
        f'Facebook ego-network — {G.number_of_nodes():,} nodes, '
        f'{G.number_of_edges():,} edges (size & colour by degree)'
    )
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as)
    plt.show()
    return pos


# Cache the layout we'll reuse it later when drawing communities and bridges
pos = visualise_network(G, save_as='figures/fig2_network.png')

Several dense blobs are clearly visible. Each one is built around one of the ten ego users sitting at the centre of their own friendship circle. Thin chains of edges connect the blobs together and these are the bridges we'll look at next when we compute edge betweenness.

## Task 2: Edge betweenness centrality (EBC) and its distribution

Edges with high EBC are *bridges* and removing one of them forces a lot of traffic through a longer detour.

Computing EBC exactly will take around $3.5 \times 10^8$ operations on this graph and would take several minutes. NetworkX provides an unbiased estimator that uses k randomly-sampled source nodes and the variance shrinks like 1/√k. With k = 400 (about 10% of nodes), the ranking of bridge edges is stable enough for our purposes while keeping the runtime under a minute (Brandes and Pich, 2007).

In [ ]:
def compute_edge_betweenness(G, k=400, seed=SEED):
    """Approximate edge betweenness using k sampled source nodes."""
    
    ebc = nx.edge_betweenness_centrality(G, k=k, seed=seed, normalized=True)
    return ebc


ebc = compute_edge_betweenness(G)

# Pull values into a numpy array for plotting and summary stats
ebc_values = np.array(list(ebc.values()))

# Quick numerical summary
print(f'mean EBC: {ebc_values.mean():.3e}')
print(f'median EBC: {np.median(ebc_values):.3e}')
print(f'max EBC: {ebc_values.max():.3e}')

# The top-10 edges are the strongest bridges
print('\nTop 10 edges by EBC:')
top10 = sorted(ebc.items(), key=lambda kv: kv[1], reverse=True)[:10]
for (u, v), score in top10:
    print(f'  ({u:>5}, {v:>5})  {score:.4e}')

In [ ]:
def plot_ebc_distribution(values, save_as=None):
    """Plot the EBC distribution on linear and log-log axes."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    # Linear panel — most edges fall near zero, so the bulk piles up on the left
    axes[0].hist(values, bins=80, color='#9b3b3b', edgecolor='white')
    axes[0].set_xlabel('Edge betweenness centrality')
    axes[0].set_ylabel('Number of edges')
    axes[0].set_title('Linear scale')

    # Log-log panel — exposes the heavy tail of bridge edges
    positive = values[values > 0]
    bins = np.logspace(np.log10(positive.min()), np.log10(positive.max()), 60)
    axes[1].hist(positive, bins=bins, color='#9b3b3b', edgecolor='white')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('EBC (log)')
    axes[1].set_ylabel('Number of edges (log)')
    axes[1].set_title('Log–log scale')

    plt.suptitle('Distribution of edge betweenness centrality')
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as)
    plt.show()


plot_ebc_distribution(ebc_values, save_as='figures/fig3_ebc_dist.png')

The distribution is extremely skewed. Almost every edge sits in a dense local
cluster, where there are many redundant short paths around it, so its EBC is
close to zero. A small minority of edges — the bridges that connect the
ten ego networks — pick up huge amounts of shortest-path traffic. The maximum
is more than three orders of magnitude above the median.

Plotting these top edges on the network makes the role of the bridges easy
to see:


In [ ]:
def highlight_bridges(G, ebc, pos, top_pct=1, save_as=None):
    """Re-draw the network with the top top_pct% of edges (by EBC) in red."""
    # Threshold corresponds to (1 - top_pct/100) quantile
    quantile = 1 - top_pct / 100.0
    threshold = np.quantile(list(ebc.values()), quantile)
    bridges = [edge for edge, score in ebc.items() if score >= threshold]

    deg = dict(G.degree())
    node_size = [max(2, 0.05 * deg[n]) for n in G.nodes()]

    plt.figure(figsize=(10, 9))
    
    # All edges first, very faint
    nx.draw_networkx_edges(G, pos, alpha=0.04, width=0.3)
    
    # Bridges on top in red
    nx.draw_networkx_edges(G, pos, edgelist=bridges, edge_color='red', width=0.9, alpha=0.85)
    
    # Nodes in dark grey so the red bridges stand out
    nx.draw_networkx_nodes(G, pos, node_size=node_size, node_color='#444', linewidths=0, alpha=0.7)
    plt.axis('off')
    plt.title(f'Top {top_pct}% of edges by betweenness ({len(bridges)} edges)')
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as)
    plt.show()
    return bridges


bridges = highlight_bridges(G, ebc, pos, top_pct=1, save_as='figures/fig4_bridges.png')

The red edges trace the inter-community pathways. They line up almost exactly with the visual gaps between the dense blobs we saw earlier. This is a nice qualitative check that the EBC computation makes sense.

## Task 3: Community detection with two algorithms

To compare two methods that take genuinely different approaches I picked:

1. **Louvain** (Blondel et al., 2008). At each step it moves each node to the neighbour community that gives the largest gain in modularity Q, then collapses the result and repeats. It is fast, works well on graphs of this size, and is the standard baseline in the literature (Fortunato and Hric, 2016).
   
3. **Label Propagation** (Raghavan, Albert and Kumara, 2007). Each node adopts the label held by the majority of its neighbours. The algorithm is near-linear in the number of edges and does not optimise modularity directly, which makes it a useful contrast to Louvain.

In [ ]:
def run_louvain(G, seed=SEED, resolution=1.0):
    """Run Louvain modularity optimisation. Returns a node->community dict,
    the modularity Q, and a sorted list of community sizes."""
    
    partition = community_louvain.best_partition(G, random_state=seed, resolution=resolution)
    Q = community_louvain.modularity(partition, G)

    # Count members per community label, then sort biggest first
    sizes = sorted(Counter(partition.values()).values(), reverse=True)

    print(f'Louvain: {len(sizes)} communities, modularity Q = {Q:.4f}')
    return partition, Q, sizes


part_lv, Q_lv, sizes_lv = run_louvain(G)
print(f'Top 10 community sizes: {sizes_lv[:10]}')
print(f'Smallest community has {sizes_lv[-1]} nodes; largest has {sizes_lv[0]}')

In [ ]:
def run_label_propagation(G, seed=SEED):
    """Run Label Propagation. Returns (partition_dict, Q, sorted_sizes)."""
    # The function uses Python's random module internally for tie-breaking, so we seed it here for reproducibility
    random.seed(seed)

    communities = list(label_propagation_communities(G))

    # Convert list-of-sets into a node->community-id dict
    partition = {node: i for i, com in enumerate(communities) for node in com}

    # Reuse the python-louvain modularity function so Q is comparable
    Q = community_louvain.modularity(partition, G)
    sizes = sorted([len(c) for c in communities], reverse=True)

    print(f'Label Propagation: {len(sizes)} communities, modularity Q = {Q:.4f}')
    return partition, Q, sizes


part_lp, Q_lp, sizes_lp = run_label_propagation(G)
print(f'Top 10 community sizes: {sizes_lp[:10]}')
print(f'Smallest community has {sizes_lp[-1]} nodes; largest has {sizes_lp[0]}')

### Side-by-side comparison

The table below summarises (a) the number of communities and (b) the size of each one, and the bar charts that follow show the full size distributions.

In [ ]:
# Build a small comparison table
summary = pd.DataFrame({
    'Algorithm': ['Louvain', 'Label Propagation'],
    'Communities found': [len(sizes_lv), len(sizes_lp)],
    'Modularity Q': [round(Q_lv, 3), round(Q_lp, 3)],
    'Largest community': [sizes_lv[0], sizes_lp[0]],
    'Smallest community': [sizes_lv[-1], sizes_lp[-1]],
    'Median size': [int(np.median(sizes_lv)), int(np.median(sizes_lp))],
})
summary

To check whether the two algorithms find a similar coarse structure even when they produce different numbers of communities, I compute two standard
partition-similarity scores: Normalised Mutual Information (NMI) and the Adjusted Rand Index (ARI). Both range roughly from 0 (no agreement) to 1
(identical partitions).

In [ ]:
def compare_partitions(part_a, part_b, G):
    """Compute NMI and ARI between two node->community label dicts."""
    # Use a fixed node order so labels line up across both partitions
    nodes = sorted(G.nodes())
    labels_a = [part_a[n] for n in nodes]
    labels_b = [part_b[n] for n in nodes]
    nmi = normalized_mutual_info_score(labels_a, labels_b)
    ari = adjusted_rand_score(labels_a, labels_b)
    return nmi, ari


nmi, ari = compare_partitions(part_lv, part_lp, G)
print(f'Normalised Mutual Information (NMI) : {nmi:.3f}')
print(f'Adjusted Rand Index (ARI)           : {ari:.3f}')

### Visualising the two partitions

Re-using the same spring layout as before lets us see directly where the two
algorithms agree and where they differ. Each colour is one community.


In [ ]:
def draw_partition(G, partition, pos, title, save_as=None):
    """Colour each node by its community label and draw the network."""
    # Look up a colour for each community id, recycling tab20 if there are more communities than colours (Label Propagation has 44, so we wrap)
    community_ids = list({partition[n] for n in G.nodes()})
    cmap = plt.cm.tab20
    colour_for = {cid: cmap(i % 20) for i, cid in enumerate(community_ids)}
    node_colours = [colour_for[partition[n]] for n in G.nodes()]

    deg = dict(G.degree())
    node_size = [max(2, 0.05 * deg[n]) for n in G.nodes()]

    plt.figure(figsize=(10, 9))
    nx.draw_networkx_edges(G, pos, alpha=0.04, width=0.3)
    nx.draw_networkx_nodes(G, pos, node_size=node_size, node_color=node_colours, linewidths=0)
    plt.axis('off')
    plt.title(title)
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as)
    plt.show()


draw_partition(G, part_lv, pos, f'Louvain communities (k={len(sizes_lv)}, Q={Q_lv:.3f})', save_as='figures/fig5_louvain.png')

draw_partition(G, part_lp, pos, f'Label Propagation communities (k={len(sizes_lp)}, Q={Q_lp:.3f})', save_as='figures/fig6_lp.png')

In [ ]:
# Bar chart comparison of community sizes
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Louvain has only around 15 communities so we can show all of them
axes[0].bar(range(len(sizes_lv)), sizes_lv, color='#3b6ea5')
axes[0].set_title(f'Louvain — community sizes (k = {len(sizes_lv)})')
axes[0].set_xlabel('Community rank (largest first)')
axes[0].set_ylabel('Number of nodes')

# Label Propagation has many small communities so we will show only the top 40
top_n = min(40, len(sizes_lp))
axes[1].bar(range(top_n), sizes_lp[:top_n], color='#9b3b3b')
axes[1].set_title(f'Label Propagation — top {top_n} of {len(sizes_lp)}')
axes[1].set_xlabel('Community rank (largest first)')
axes[1].set_ylabel('Number of nodes')

plt.tight_layout()
plt.savefig('figures/fig7_community_sizes.png')
plt.show()

The bar charts make the difference clear. Louvain returns around 15 large blocks with a roughly even spread of sizes. Label Propagation finds a couple of very large communities and then a long tail of smaller ones. Several of them only contain 2 to 3 nodes.